In [1]:
# create daily regression fit for predictor features - insitu observation

# filter for observations on class 1 - rain >= 0.1 mm

# WORKFLOW
# 1. Load dfs and format for fitting
# 2. Perform cluster*season wise ML fit
# 3. Check if cluster stratification improves the model
# 4. Check if season stratification improves the model 
# 5. Saved trained regressor model using joblib

In [ ]:
# Libraries
import os
import numpy as np  
import pandas as pd
import joblib

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

# Paths
train_df_path = r"..\Data\Processed\Bias_correction_daily\Prepared_dataframe\final_train_df.parquet"
test_df_path = r"..\Data\Processed\Bias_correction_daily\Prepared_dataframe\final_test_df.parquet"
regressor_model_path = r"..\Data\Processed\Bias_correction_daily\Regressor_model\saved_regressors"
regressor_metrics_path = r"..\Data\Processed\Bias_correction_daily\Regressor_model\validation_results"
rscv_dir = r"..\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results"

os.makedirs(regressor_model_path, exist_ok = True)
os.makedirs(regressor_metrics_path, exist_ok = True)
os.makedirs(rscv_dir, exist_ok = True)

# Configurations
random_state_num = 64
n_cores = 2
target_variable = 'insitu_prec'
predictor_features = ['y', 'x', 
        'era5l', 'chirps', 'har', 
        'persiann_prec', 'mswep_prec', 
        'gldas_air_temp', 'gldas_lwdown', 'gldas_specific_humidity',
        'gldas_surface_pressure', 'gldas_swdown', 'gldas_wind_speed',
        'insitu_kriged', 'cluster', 'season', 'season_num',
        'elev_mean', 'elev_std', 'elev_max', 'elev_min', 'slope_mean', 
        'aspect_mean', 'elev_smoothed', 'fetch_1000m_dist', 'fetch_1000m_dir',
        'fetch_500m_dist', 'fetch_500m_dir', 'local_relief', 'doy_sin', 'doy_cos']

In [3]:
SEASONS = {
    'winter':       [12, 1, 2],
    'pre_monsoon':  [3, 4, 5],
    'monsoon':      [6, 7, 8, 9],
    'post_monsoon': [10, 11],
}

def get_season(month):
    for s, months in SEASONS.items():
        if month in months:
            return s
    return None

SEASONS_NUM = {
    1: 'winter',
    2: 'pre_monsoon',
    3: 'monsoon',
    4: 'post_monsoon'
}

def get_season_num(season_str):
    for s_n, s_name in SEASONS_NUM.items():
        if s_name == season_str:
            return s_n
    return None

# Helper functions
def get_param_grids():
    param_dist_rf = {
        'n_estimators':      [100, 1000, 3000, 5000],
        'max_features':      ['sqrt', 'log2'],
        'max_depth':         [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf':  [1, 2, 4],
    }
    param_dist_xgb = {
        'n_estimators':     [100, 500, 1000, 3000],
        'max_depth':        [3, 5, 7, 10],
        'learning_rate':    [0.01, 0.05, 0.1, 0.2],
        'subsample':        [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'gamma':            [0, 0.1, 0.5],
    }
    
    return param_dist_rf, param_dist_xgb

def get_param_grids_increased():
    param_dist_rf = {
        'n_estimators':      [7000, 10000, 15000],
        'max_features':      ['sqrt', 'log2'],
        'max_depth':         [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf':  [1, 2, 4],
    }
    param_dist_xgb = {
        'n_estimators':     [5000, 7000, 10000],
        'max_depth':        [3, 5, 7, 10],
        'learning_rate':    [0.01, 0.05, 0.1, 0.2],
        'subsample':        [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'gamma':            [0, 0.1, 0.5],
    }
    
    return param_dist_rf, param_dist_xgb

def compute_metrics_regression(y_true, y_pred, label):
    """Compute RMSE, MAE, R² and print with label."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"    [{label}]  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}  n={len(y_true)}")
    return {'rmse': round(rmse, 4), 'mae': round(mae, 4), 'r2': round(r2, 4),
            'n': len(y_true)}

In [4]:
# 1. Load and format dfs
train_df = pd.read_parquet(train_df_path)
test_df = pd.read_parquet(test_df_path)

seasons_train_da = train_df['month'].apply(get_season)
train_df['season'] = seasons_train_da
seasons_test_da = test_df['month'].apply(get_season)
test_df['season'] = seasons_test_da

train_df['season_num'] = train_df['season'].apply(get_season_num)
test_df['season_num'] = test_df['season'].apply(get_season_num)

train_df = train_df[train_df['insitu_prec'] >= 0.1]
test_df = test_df[test_df['insitu_prec'] >= 0.1]

cluster_list = sorted(test_df['cluster'].unique().tolist())
season_list = list(SEASONS)

In [14]:
# 2. First we perform a full cluster*season stratified fit

for cluster in cluster_list:
    cluster_train = train_df[train_df['cluster'] == cluster].copy()
    cluster_test = test_df[test_df['cluster'] == cluster].copy()

    for season in season_list:
        print(f"Processing Cluster {cluster} & Season {season} -----------------------------")

        season_cluster_train = cluster_train[cluster_train['season'] == season].copy()
        season_cluster_test = cluster_test[cluster_test['season'] == season].copy()
        
        X_train = season_cluster_train[predictor_features].drop(columns= ['cluster', 'season', 'season_num'], errors='ignore').values
        y_train = season_cluster_train[target_variable].values

        if (cluster == 0.0 and season == 'monsoon'):
            param_dist_rf, _ = get_param_grids_increased()
        elif (cluster == 1.0 and season in ['pre_monsoon', 'monsoon']):
            param_dist_rf, _ = get_param_grids_increased()
        elif (cluster == 2.0 and season in ['pre_monsoon', 'winter']):
            param_dist_rf, _ = get_param_grids_increased()
        elif (cluster == 3.0 and season in ['pre_monsoon', 'monsoon']):
            param_dist_rf, _ = get_param_grids_increased()
        elif (cluster == 3.0 and season == 'post_monsoon'):
            _, param_dist_xgb = get_param_grids_increased()                    
        else:
            param_dist_rf, param_dist_xgb = get_param_grids()

        for model_type in ['rf', 'xgb']:
            if model_type == 'rf':
                base_model = Pipeline([
                    ('imputer', SimpleImputer(strategy = 'median')),
                    ('regressor', RandomForestRegressor(random_state = random_state_num,
                        n_jobs = n_cores))
                ])
                param_dist = {f'regressor__{k}': v for k, v in param_dist_rf.items()}
            else:
                base_model = Pipeline([
                    ('imputer', SimpleImputer(strategy = 'median')),
                    ('regressor',     XGBRegressor(random_state=random_state_num,
                                    n_jobs=n_cores,
                                    verbosity = 0)),    
                ])
                param_dist = {f'regressor__{k}': v for k, v in param_dist_xgb.items()}

            search_cv = RandomizedSearchCV(
                estimator = base_model,
                param_distributions = param_dist,
                n_iter = 30,
                cv = 3,
                verbose = 1,
                n_jobs = n_cores,
                scoring = 'neg_root_mean_squared_error'
            )
            search_cv.fit(X_train, y_train)
            best_model = search_cv.best_estimator_
            rscv_results_df = pd.DataFrame(search_cv.cv_results_)
            rscv_path = os.path.join(rscv_dir, f"{cluster}_{season}_{model_type}.csv")
            rscv_results_df.to_csv(rscv_path)

            print(f"\n  [Cluster {cluster} | {model_type.upper()}]")
            print(f"    Best params : {search_cv.best_params_}")
            print(f"    CV RMSE     : {search_cv.best_score_:.4f}")
            print(f"RSCV results saved to : {rscv_path}")
            
            model_path = os.path.join(regressor_model_path, f"{cluster}_{season}_{model_type}.pkl")
            joblib.dump(best_model, model_path)
            print(f"Model  {model_type} saved to {model_path}")

Processing Cluster 0.0 & Season winter -----------------------------
Fitting 3 folds for each of 30 candidates, totalling 90 fits

  [Cluster 0.0 | RF]
    Best params : {'regressor__n_estimators': 5000, 'regressor__min_samples_split': 5, 'regressor__min_samples_leaf': 2, 'regressor__max_features': 'sqrt', 'regressor__max_depth': None}
    CV RMSE     : -12.8444
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results\0.0_winter_rf.csv
Model  rf saved to ..\Data\Processed\Bias_correction_daily\Regressor_model\saved_classifiers\0.0_winter_rf.pkl
Fitting 3 folds for each of 30 candidates, totalling 90 fits

  [Cluster 0.0 | XGB]
    Best params : {'regressor__subsample': 0.8, 'regressor__n_estimators': 1000, 'regressor__max_depth': 5, 'regressor__learning_rate': 0.05, 'regressor__gamma': 0.1, 'regressor__colsample_bytree': 1.0}
    CV RMSE     : -10.7570
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results\0.0_wi

c:\Users\A S U S\.conda\envs\geo_stack\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



  [Cluster 1.0 | RF]
    Best params : {'regressor__n_estimators': 7000, 'regressor__min_samples_split': 2, 'regressor__min_samples_leaf': 1, 'regressor__max_features': 'sqrt', 'regressor__max_depth': 20}
    CV RMSE     : -7.3947
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results\1.0_pre_monsoon_rf.csv
Model  rf saved to ..\Data\Processed\Bias_correction_daily\Regressor_model\saved_classifiers\1.0_pre_monsoon_rf.pkl
Fitting 3 folds for each of 30 candidates, totalling 90 fits

  [Cluster 1.0 | XGB]
    Best params : {'regressor__subsample': 0.8, 'regressor__n_estimators': 1000, 'regressor__max_depth': 5, 'regressor__learning_rate': 0.01, 'regressor__gamma': 0.5, 'regressor__colsample_bytree': 1.0}
    CV RMSE     : -6.7358
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results\1.0_pre_monsoon_xgb.csv
Model  xgb saved to ..\Data\Processed\Bias_correction_daily\Regressor_model\saved_classifiers\1.0_pre_mons

c:\Users\A S U S\.conda\envs\geo_stack\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



  [Cluster 3.0 | RF]
    Best params : {'regressor__n_estimators': 10000, 'regressor__min_samples_split': 2, 'regressor__min_samples_leaf': 1, 'regressor__max_features': 'sqrt', 'regressor__max_depth': 30}
    CV RMSE     : -3.3663
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results\3.0_monsoon_rf.csv
Model  rf saved to ..\Data\Processed\Bias_correction_daily\Regressor_model\saved_classifiers\3.0_monsoon_rf.pkl
Fitting 3 folds for each of 30 candidates, totalling 90 fits

  [Cluster 3.0 | XGB]
    Best params : {'regressor__subsample': 1.0, 'regressor__n_estimators': 1000, 'regressor__max_depth': 3, 'regressor__learning_rate': 0.01, 'regressor__gamma': 0, 'regressor__colsample_bytree': 1.0}
    CV RMSE     : -2.0887
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results\3.0_monsoon_xgb.csv
Model  xgb saved to ..\Data\Processed\Bias_correction_daily\Regressor_model\saved_classifiers\3.0_monsoon_xgb.pkl
Proce

In [15]:
# 3. Cluster wise only fit

for cluster in cluster_list:
    print(f"\n Processing Cluster {cluster}-----------------------------")
    cluster_train = train_df[train_df['cluster'] == cluster].copy()
    cluster_test = test_df[test_df['cluster'] == cluster].copy()
        
    X_train = cluster_train[predictor_features].drop(columns= ['cluster', 'season'], errors='ignore').values
    y_train = cluster_train[target_variable].values

    param_dist_rf, param_dist_xgb = get_param_grids()

    for model_type in ['rf', 'xgb']:
        if model_type == 'rf':
            base_model = Pipeline([
                ('imputer', SimpleImputer(strategy = 'median')),
                ('regressor', RandomForestRegressor(random_state = random_state_num,
                    n_jobs = n_cores))
            ])
            param_dist = {f'regressor__{k}': v for k, v in param_dist_rf.items()}
        else:
            base_model = Pipeline([
                ('imputer', SimpleImputer(strategy = 'median')),
                ('regressor',     XGBRegressor(random_state=random_state_num,
                                n_jobs=n_cores,
                                verbosity = 0)),    
            ])
            param_dist = {f'regressor__{k}': v for k, v in param_dist_xgb.items()}

        search_cv = RandomizedSearchCV(
            estimator = base_model,
            param_distributions = param_dist,
            n_iter = 30,
            cv = 3,
            verbose = 1,
            n_jobs = n_cores,
            scoring = 'neg_root_mean_squared_error'
        )
        search_cv.fit(X_train, y_train)
        best_model = search_cv.best_estimator_
        rscv_results_df = pd.DataFrame(search_cv.cv_results_)
        rscv_path = os.path.join(rscv_dir, f"{cluster}_allSeasons_{model_type}.csv")
        rscv_results_df.to_csv(rscv_path)

        print(f"\n  [Cluster {cluster} | {model_type.upper()}]")
        print(f"    Best params : {search_cv.best_params_}")
        print(f"    CV RMSE     : {search_cv.best_score_:.4f}")
        print(f"RSCV results saved to : {rscv_path}")
            
        model_path = os.path.join(regressor_model_path, f"{cluster}_allSeasons_{model_type}.pkl")
        joblib.dump(best_model, model_path)
        print(f"Model  {model_type} saved to {model_path}")


 Processing Cluster 0.0-----------------------------
Fitting 3 folds for each of 30 candidates, totalling 90 fits

  [Cluster 0.0 | RF]
    Best params : {'regressor__n_estimators': 5000, 'regressor__min_samples_split': 2, 'regressor__min_samples_leaf': 1, 'regressor__max_features': 'sqrt', 'regressor__max_depth': None}
    CV RMSE     : -10.6914
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results\0.0_allSeasons_rf.csv
Model  rf saved to ..\Data\Processed\Bias_correction_daily\Regressor_model\saved_classifiers\0.0_allSeasons_rf.pkl
Fitting 3 folds for each of 30 candidates, totalling 90 fits

  [Cluster 0.0 | XGB]
    Best params : {'regressor__subsample': 0.6, 'regressor__n_estimators': 100, 'regressor__max_depth': 3, 'regressor__learning_rate': 0.05, 'regressor__gamma': 0.5, 'regressor__colsample_bytree': 0.8}
    CV RMSE     : -8.8348
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results\0.0_allSeasons_

c:\Users\A S U S\.conda\envs\geo_stack\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



  [Cluster 2.0 | RF]
    Best params : {'regressor__n_estimators': 5000, 'regressor__min_samples_split': 2, 'regressor__min_samples_leaf': 1, 'regressor__max_features': 'sqrt', 'regressor__max_depth': None}
    CV RMSE     : -12.1955
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results\2.0_allSeasons_rf.csv
Model  rf saved to ..\Data\Processed\Bias_correction_daily\Regressor_model\saved_classifiers\2.0_allSeasons_rf.pkl
Fitting 3 folds for each of 30 candidates, totalling 90 fits

  [Cluster 2.0 | XGB]
    Best params : {'regressor__subsample': 0.6, 'regressor__n_estimators': 1000, 'regressor__max_depth': 5, 'regressor__learning_rate': 0.01, 'regressor__gamma': 0, 'regressor__colsample_bytree': 0.8}
    CV RMSE     : -8.9914
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results\2.0_allSeasons_xgb.csv
Model  xgb saved to ..\Data\Processed\Bias_correction_daily\Regressor_model\saved_classifiers\2.0_allSeasons

In [7]:
# 4. Season wise only fit
for season in season_list:
    print(f"\n Processing Season {season}-----------------------------")
    season_train = train_df[train_df['season'] == season].copy()
    season_test = test_df[test_df['season'] == season].copy()
        
    X_train = season_train[predictor_features].drop(columns= ['season', 'season_num'], errors='ignore').values
    y_train = season_train[target_variable].values

    if season in ['monsoon', 'post_monsoon', 'winter']:
            param_dist_rf, _ = get_param_grids_increased()
            _, param_dist_xgb = get_param_grids()    
    else:
        param_dist_rf, param_dist_xgb = get_param_grids()
    for model_type in ['rf', 'xgb']:
        if model_type == 'rf':
            base_model = Pipeline([
                ('imputer', SimpleImputer(strategy = 'median')),
                ('regressor', RandomForestRegressor(random_state = random_state_num,
                    n_jobs = n_cores))
            ])
            param_dist = {f'regressor__{k}': v for k, v in param_dist_rf.items()}
        else:
            base_model = Pipeline([
                ('imputer', SimpleImputer(strategy = 'median')),
                ('regressor',     XGBRegressor(random_state=random_state_num,
                                n_jobs=n_cores,
                                verbosity = 0)),    
            ])
            param_dist = {f'regressor__{k}': v for k, v in param_dist_xgb.items()}

        search_cv = RandomizedSearchCV(
            estimator = base_model,
            param_distributions = param_dist,
            n_iter = 30,
            cv = 3,
            verbose = 1,
            n_jobs = n_cores,
            scoring = 'neg_root_mean_squared_error'
        )
        search_cv.fit(X_train, y_train)
        best_model = search_cv.best_estimator_
        rscv_results_df = pd.DataFrame(search_cv.cv_results_)
        rscv_path  = os.path.join(rscv_dir, f"{9.0}_{season}_{model_type}.csv")
        rscv_results_df.to_csv(rscv_path)

        print(f"\n  [Cluster {season} | {model_type.upper()}]")
        print(f"    Best params : {search_cv.best_params_}")
        print(f"    CV RMSE     : {search_cv.best_score_:.4f}")
        print(f"RSCV results saved to : {rscv_path}")
            
        model_path = os.path.join(regressor_model_path, f"{9.0}_{season}_{model_type}.pkl")
        joblib.dump(best_model, model_path)
        print(f"Model  {model_type} saved to {model_path}")


 Processing Season winter-----------------------------
Fitting 3 folds for each of 30 candidates, totalling 90 fits


c:\Users\A S U S\.conda\envs\geo_stack\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



  [Cluster winter | RF]
    Best params : {'regressor__n_estimators': 10000, 'regressor__min_samples_split': 2, 'regressor__min_samples_leaf': 2, 'regressor__max_features': 'sqrt', 'regressor__max_depth': 30}
    CV RMSE     : -8.8601
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results\9.0_winter_rf.csv
Model  rf saved to ..\Data\Processed\Bias_correction_daily\Regressor_model\saved_classifiers\9.0_winter_rf.pkl
Fitting 3 folds for each of 30 candidates, totalling 90 fits

  [Cluster winter | XGB]
    Best params : {'regressor__subsample': 0.8, 'regressor__n_estimators': 100, 'regressor__max_depth': 3, 'regressor__learning_rate': 0.05, 'regressor__gamma': 0.1, 'regressor__colsample_bytree': 0.8}
    CV RMSE     : -7.6652
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results\9.0_winter_xgb.csv
Model  xgb saved to ..\Data\Processed\Bias_correction_daily\Regressor_model\saved_classifiers\9.0_winter_xgb.pkl

 

c:\Users\A S U S\.conda\envs\geo_stack\Lib\site-packages\joblib\externals\loky\process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



  [Cluster post_monsoon | RF]
    Best params : {'regressor__n_estimators': 7000, 'regressor__min_samples_split': 5, 'regressor__min_samples_leaf': 1, 'regressor__max_features': 'sqrt', 'regressor__max_depth': 20}
    CV RMSE     : -13.0951
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results\9.0_post_monsoon_rf.csv
Model  rf saved to ..\Data\Processed\Bias_correction_daily\Regressor_model\saved_classifiers\9.0_post_monsoon_rf.pkl
Fitting 3 folds for each of 30 candidates, totalling 90 fits

  [Cluster post_monsoon | XGB]
    Best params : {'regressor__subsample': 0.8, 'regressor__n_estimators': 500, 'regressor__max_depth': 3, 'regressor__learning_rate': 0.01, 'regressor__gamma': 0, 'regressor__colsample_bytree': 0.8}
    CV RMSE     : -11.8178
RSCV results saved to : ..\Data\Processed\Bias_correction_daily\Regressor_model\rscv_results\9.0_post_monsoon_xgb.csv
Model  xgb saved to ..\Data\Processed\Bias_correction_daily\Regressor_model\saved_clas

In [6]:
# 4. Aggregated fit

train = train_df.copy()
test = test_df.copy()
        
X_train = train[predictor_features].drop(columns= ['season'], errors='ignore').values
y_train = train[target_variable].values

param_dist_rf, param_dist_xgb = get_param_grids()

for model_type in ['rf', 'xgb']:
    if model_type == 'rf':
        base_model = Pipeline([
            ('imputer', SimpleImputer(strategy = 'median')),
            ('regressor', RandomForestRegressor(random_state = random_state_num,
                n_jobs = n_cores))
        ])
        param_dist = {f'regressor__{k}': v for k, v in param_dist_rf.items()}
    else:
        base_model = Pipeline([
            ('imputer', SimpleImputer(strategy = 'median')),
            ('regressor',     XGBRegressor(random_state=random_state_num,
                            n_jobs=n_cores,
                            verbosity = 0)),    
        ])
        param_dist = {f'regressor__{k}': v for k, v in param_dist_xgb.items()}

    search_cv = RandomizedSearchCV(
        estimator = base_model,
        param_distributions = param_dist,
        n_iter = 30,
        cv = 3,
        verbose = 1,
        n_jobs = n_cores,
        scoring = 'neg_root_mean_squared_error'
    )
    search_cv.fit(X_train, y_train)
    best_model = search_cv.best_estimator_
    rscv_results_df = pd.DataFrame(search_cv.cv_results_)
    rscv_path = os.path.join(rscv_dir, f"{9.0}_allSeasons_{model_type}.csv")
    rscv_results_df.to_csv(rscv_path)

    print(f"\n  [Cluster {9.0} | {model_type.upper()}]")
    print(f"    Best params : {search_cv.best_params_}")
    print(f"    CV RMSE     : {search_cv.best_score_:.4f}")
    print(f"RSCV results saved to : {rscv_path}")
            
    model_path = os.path.join(regressor_model_path, f"{9.0}_allSeasons_{model_type}.pkl")
    joblib.dump(best_model, model_path)
    print(f"Model  {model_type} saved to {model_path}")

Fitting 3 folds for each of 30 candidates, totalling 90 fits


KeyboardInterrupt: 

In [8]:
# %%
# 5. Validation: per cluster*season subset, compare all four model strategies

# Feature subsets matching how each strategy was trained
feature_sets = {
    'cluster_season': [f for f in predictor_features if f not in ('cluster', 'season', 'season_num')],
    'cluster_only':   [f for f in predictor_features if f not in ('cluster', 'season')],
    'season_only':    [f for f in predictor_features if f not in ('season', 'season_num')],
    'aggregated':     [f for f in predictor_features if f not in ('season',)],
}

# Model path templates per strategy
def get_model_path(strategy, cluster, season, model_type):
    templates = {
        'cluster_season': f"{cluster}_{season}_{model_type}.pkl",
        'cluster_only':   f"{cluster}_allSeasons_{model_type}.pkl",
        'season_only':    f"{9.0}_{season}_{model_type}.pkl",   # NOTE: season-only loop has a
        'aggregated':     f"{9.0}_allSeasons_{model_type}.pkl", #  bug saving as cluster_{mt}.pkl;
    }                                                      #  fix that loop's save path too
    return os.path.join(regressor_model_path, templates[strategy])

all_metrics = []

for cluster in cluster_list:
    cluster_test = test_df[test_df['cluster'] == cluster].copy()

    for season in season_list:
        season_cluster_test = cluster_test[cluster_test['season'] == season].copy()

        if season_cluster_test.empty:
            print(f"[SKIP] No test data — Cluster {cluster} | Season {season}")
            continue

        y_test = season_cluster_test[target_variable].values

        for model_type in ['rf', 'xgb']:
            for strategy, features in feature_sets.items():
                model_path = get_model_path(strategy, cluster, season, model_type)

                if not os.path.exists(model_path):
                    print(f"  [MISSING] {model_path}")
                    continue

                model    = joblib.load(model_path)
                X_test   = season_cluster_test[features].values
                y_pred   = model.predict(X_test)

                label    = f"C{cluster} | {season:<13} | {model_type.upper()} | {strategy}"
                metrics  = compute_metrics_regression(y_test, y_pred, label)
                metrics.update({
                    'cluster':    cluster,
                    'season':     season,
                    'model_type': model_type,
                    'strategy':   strategy,
                })
                all_metrics.append(metrics)

# %%
# Collate and save
metrics_df = pd.DataFrame(all_metrics)

# Pivot for easy visual comparison: rows = cluster*season, cols = strategy*model_type
pivot = metrics_df.pivot_table(
    index   = ['cluster', 'season'],
    columns = ['model_type', 'strategy'],
    values  = ['rmse', 'r2'],
)
pivot.columns = ['_'.join(c) for c in pivot.columns]
pivot = pivot.reset_index()

metrics_out_path = os.path.join(regressor_metrics_path, "validation_comparison.csv")
pivot_out_path   = os.path.join(regressor_metrics_path, "validation_pivot.csv")

metrics_df.to_csv(metrics_out_path, index=False)
pivot.to_csv(pivot_out_path,        index=False)

print(f"\nAll metrics  → {metrics_out_path}")
print(f"Pivot table  → {pivot_out_path}\n")
print(pivot.to_string(index=False))

    [C0.0 | winter        | RF | cluster_season]  RMSE=10.3029  MAE=7.4357  R²=0.3802  n=247
    [C0.0 | winter        | RF | cluster_only]  RMSE=10.3480  MAE=7.4269  R²=0.3748  n=247
    [C0.0 | winter        | RF | season_only]  RMSE=10.0124  MAE=7.1005  R²=0.4147  n=247
    [C0.0 | winter        | RF | aggregated]  RMSE=10.2400  MAE=7.2565  R²=0.3878  n=247
    [C0.0 | winter        | XGB | cluster_season]  RMSE=10.1308  MAE=6.5826  R²=0.4008  n=247
    [C0.0 | winter        | XGB | cluster_only]  RMSE=10.3866  MAE=7.0003  R²=0.3701  n=247
    [C0.0 | winter        | XGB | season_only]  RMSE=10.0799  MAE=6.7815  R²=0.4068  n=247
    [C0.0 | winter        | XGB | aggregated]  RMSE=10.2487  MAE=6.8846  R²=0.3867  n=247
    [C0.0 | pre_monsoon   | RF | cluster_season]  RMSE=10.4673  MAE=7.4327  R²=0.0966  n=519
    [C0.0 | pre_monsoon   | RF | cluster_only]  RMSE=10.4038  MAE=7.3919  R²=0.1075  n=519
    [C0.0 | pre_monsoon   | RF | season_only]  RMSE=10.4792  MAE=7.4244  R²=0.0945  n=

In [9]:
# Create a combined identifier for model and strategy
metrics_df['model_strategy'] = metrics_df['model_type'] + ' | ' + metrics_df['strategy']

# 1. Rank models within each (cluster, season) sub-problem
# We rank based on f1-score, accuracy, and a composite score (average of f1 and accuracy)
# ascending=False means the highest score gets rank 1.0
metrics_df['r2_rank'] = metrics_df.groupby(['cluster', 'season'])['r2'].rank(method='min', ascending=False)



# 2. Aggregate across all clusters and seasons to find the overall best combination
overall_ranking = metrics_df.groupby('model_strategy').agg(
    mean_r2=('r2', 'mean'),
    mean_r2_rank=('r2_rank', 'mean')
).reset_index()

# 3. Sort by mean composite rank (lowest rank number is best)
overall_ranking = overall_ranking.sort_values(by='mean_r2_rank')

# Display the final ranking
print("Overall Model x Strategy Ranking:")
display(overall_ranking)

# Optional: Save the overall ranking alongside the other validation metrics
ranking_out_path = os.path.join(regressor_metrics_path, "overall_model_ranking.csv")
overall_ranking.to_csv(ranking_out_path, index=False)
print(f"\nSaved overall ranking to -> {ranking_out_path}")

Overall Model x Strategy Ranking:


,model_strategy,mean_r2,mean_r2_rank
3,rf | season_only,0.161350,2.0625
2,rf | cluster_season,0.165725,2.6250
0,rf | aggregated,0.137031,3.2500
1,rf | cluster_only,0.132175,3.6875
7,xgb | season_only,0.102206,5.6250
5,xgb | cluster_only,0.088737,6.0000
6,xgb | cluster_season,0.072762,6.0625
4,xgb | aggregated,0.083687,6.6875



Saved overall ranking to -> ..\Data\Processed\Bias_correction_daily\Regressor_model\validation_results\overall_model_ranking.csv
